# ISOM5240 Fine-tuning Notebook — Pipeline 1: Shelf-life Classification

**Workflow:**
- Phase 1 → Compare 3 pre-trained models via keyword mapping (NO training) → select best
- Phase 2 → Fine-tune only the selected model on full dataset
- Phase 3 → Evaluate + export Excel + push to HuggingFace Hub

**Datasets:**
- Grocery Store Dataset (GitHub) → short_shelf + medium_shelf
- Kaggle Household Products → non_perishable

**Task:** 3-class image classification (short_shelf / medium_shelf / non_perishable)

## Step 1: Install dependencies

In [1]:
!pip install transformers datasets evaluate accelerate pillow scikit-learn -q
!pip install huggingface_hub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00


## Step 2: GPU check

In [2]:
import torch
import os
import time
import numpy as np
import glob
from PIL import Image

if not torch.cuda.is_available():
    print("No GPU! Runtime → Change runtime type → T4 GPU → Save, then Run All.")
else:
  print(f"Using GPU: {torch.cuda.get_device_name(0)}")

Using GPU: Tesla T4


## Step 3: Login to HuggingFace + setup Kaggle

In [3]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
print("HuggingFace + Kaggle ready")

HuggingFace + Kaggle ready


## Step 4: Download both datasets

In [4]:
!git clone https://github.com/marcusklasson/GroceryStoreDataset.git
!kaggle datasets download -d taru149/householdproducts
!unzip -q householdproducts.zip -d household/
print("Both datasets downloaded!")

Cloning into 'GroceryStoreDataset'...
remote: Enumerating objects: 6559, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (229/229), done.
remote: Total 6559 (delta 45), reused 35 (delta 35), pack-reused 6295 (from 2)
Receiving objects: 100% (6559/6559), 116.26 MiB | 19.55 MiB/s, done.
Resolving deltas: 100% (275/275), done.
Dataset URL: https://www.kaggle.com/datasets/taru149/householdproducts
License(s): unknown
100% 33.4M/33.4M [00:03<00:00, 9.86MB/s]

Both datasets downloaded!


## Step 5: Build 3-class dataset

- short_shelf (0): fruits, vegetables
- medium_shelf (1): dairy, juice, milk
- non_perishable (2): household products

In [5]:
SHORT_KEYWORDS = [
    "apple", "avocado", "banana", "kiwi", "lemon", "lime", "mango",
    "melon", "nectarine", "orange", "papaya", "passion", "peach",
    "pear", "pineapple", "plum", "pomegranate", "grapefruit",
    "satsuma", "watermelon", "asparagus", "aubergine", "cabbage",
    "carrot", "cucumber", "garlic", "ginger", "leek", "mushroom",
    "onion", "pepper", "potato", "red-beet", "tomato", "zucchini"
]

MEDIUM_KEYWORDS = [
    "juice", "milk", "oat", "sour-cream", "sour-milk",
    "soy", "yoghurt", "cream"
]

LABEL_NAMES = ["short_shelf", "medium_shelf", "non_perishable"]

def collect_grocery_images(split_dir):
    images, labels = [], []
    for root, dirs, files in os.walk(split_dir):
        folder_name = os.path.basename(root).lower()

        if any(kw in folder_name for kw in SHORT_KEYWORDS):
            label = 0
        elif any(kw in folder_name for kw in MEDIUM_KEYWORDS):
            label = 1
        else:
            continue

        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                images.append(os.path.join(root, f))
                labels.append(label)

    return images, labels

grocery_train_imgs, grocery_train_labels = collect_grocery_images("GroceryStoreDataset/dataset/train")
grocery_test_imgs, grocery_test_labels = collect_grocery_images("GroceryStoreDataset/dataset/test")

household_imgs = (glob.glob("household/**/*.jpg", recursive=True) +
                  glob.glob("household/**/*.png", recursive=True) +
                  glob.glob("household/**/*.jpeg", recursive=True))

print(f"Grocery train: {len(grocery_train_imgs)} | Grocery test: {len(grocery_test_imgs)}")
print(f"Household: {len(household_imgs)}")

Grocery train: 2187 | Grocery test: 2042
Household: 348


In [6]:
from sklearn.model_selection import train_test_split

if len(household_imgs) > 0:
    hh_train, hh_test = train_test_split(household_imgs, test_size=0.2, random_state=42)
else:
    hh_train, hh_test = [], []

all_train_paths = grocery_train_imgs + hh_train
all_train_labels = grocery_train_labels + [2] * len(hh_train)

all_test_paths = grocery_test_imgs + hh_test
all_test_labels = grocery_test_labels + [2] * len(hh_test)

all_train_labels_np = np.array(all_train_labels)
all_test_labels_np = np.array(all_test_labels)

print(f"Combined train: {len(all_train_paths)} | Combined test: {len(all_test_paths)}")
for i, name in enumerate(LABEL_NAMES):
    print(f"  {name}: train={int((all_train_labels_np==i).sum())}, test={int((all_test_labels_np==i).sum())}")

Combined train: 2465 | Combined test: 2112
  short_shelf: train=1544, test=1456
  medium_shelf: train=643, test=586
  non_perishable: train=278, test=70


---
# PHASE 1: Model Selection (pre-trained + keyword mapping, NO training)

Compare 3 models using ImageNet labels mapped to shelf-life categories via keywords.
Same approach as the Pipeline 1 comparison notebook.

## Step 6: Define candidate models and keyword mapping

In [7]:
CANDIDATE_MODELS = {
    "ViT-base": "google/vit-base-patch16-224",
    "ResNet-50": "microsoft/resnet-50",
    "Swin-tiny": "microsoft/swin-tiny-patch4-window7-224",
}

# ImageNet label → shelf-life category
IMAGENET_SHORT = [
    "banana", "orange", "strawberry", "apple", "lemon", "pineapple",
    "pomegranate", "fig", "jackfruit", "mango", "broccoli", "cucumber",
    "mushroom", "meat", "egg", "bakery", "bread", "grocery", "fruit",
    "vegetable", "food", "pizza", "hotdog", "pretzel", "bagel", "dough",
    "zucchini", "pepper", "cauliflower", "artichoke", "potato", "cabbage",
    "head cabbage", "corn", "acorn squash", "spaghetti squash",
    "butternut squash", "ice cream", "custard apple"
]

IMAGENET_MEDIUM = [
    "bottle", "can", "jar", "packet", "carton", "sauce", "wine",
    "beer", "juice", "water", "pop", "cup", "coffee", "espresso",
    "milk can", "water bottle", "wine bottle", "beer bottle", "pop bottle"
]

def map_to_shelf_life(imagenet_label):
    label = imagenet_label.lower()
    if any(kw in label for kw in IMAGENET_SHORT):
        return 0
    elif any(kw in label for kw in IMAGENET_MEDIUM):
        return 1
    else:
        return 2  # default: non_perishable

print("Candidates and keyword mapping ready")

Candidates and keyword mapping ready


## Step 7: Compare 3 models (pre-trained, no training)

In [8]:
from transformers import pipeline as hf_pipeline
import pandas as pd

def evaluate_pretrained_p1(model_key, model_path, test_images, test_labels):
    print(f"\nEvaluating: {model_key}")

    pipe = hf_pipeline("image-classification", model=model_path, device=0)
    total_params = sum(p.numel() for p in pipe.model.parameters())

    correct = 0
    total = len(test_images)
    inference_times = []

    for i in range(total):
        img = Image.open(test_images[i]).convert("RGB")
        t0 = time.time()
        result = pipe(img, top_k=1)
        inference_times.append(time.time() - t0)

        pred = map_to_shelf_life(result[0]["label"])
        if pred == test_labels[i]:
            correct += 1

    accuracy = correct / total
    avg_ms = np.mean(inference_times) * 1000

    print(f"  Accuracy: {accuracy:.4f} | Speed: {avg_ms:.1f}ms | Params: {total_params/1e6:.1f}M")

    return {
        "Model": model_key,
        "Parameters (M)": f"{total_params/1e6:.1f}M",
        "Accuracy (pre-trained)": round(accuracy, 4),
        "Avg Inference (ms)": round(avg_ms, 1),
        "Test Samples": total,
    }

pretrained_results = []
for key, path in CANDIDATE_MODELS.items():
    r = evaluate_pretrained_p1(key, path, all_test_paths, all_test_labels)
    pretrained_results.append(r)

df_selection = pd.DataFrame(pretrained_results)
print("\n" + "="*60)
print("PHASE 1 RESULTS: Pre-trained Model Comparison")
print("="*60)
print(df_selection.to_string(index=False))


Evaluating: ViT-base


config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/113M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


  Accuracy: 0.6955 | Speed: 22.5ms | Params: 28.3M

PHASE 1 RESULTS: Pre-trained Model Comparison
    Model Parameters (M)  Accuracy (pre-trained)  Avg Inference (ms)  Test Samples
 ViT-base          86.6M                  0.6605                29.7          2112
ResNet-50          25.6M                  0.6515                13.1          2112
Swin-tiny          28.3M                  0.6955                22.5          2112


## Step 8: Select best model

In [9]:
best = max(pretrained_results, key=lambda x: x["Accuracy (pre-trained)"])
SELECTED_MODEL_KEY = best["Model"]
SELECTED_MODEL_PATH = CANDIDATE_MODELS[SELECTED_MODEL_KEY]

print(f"Selected: {SELECTED_MODEL_KEY}")
print(f"  Pre-trained accuracy: {best['Accuracy (pre-trained)']}")
print(f"  Inference speed: {best['Avg Inference (ms)']}ms")

Selected: Swin-tiny
  Pre-trained accuracy: 0.6955
  Inference speed: 22.5ms


---
# PHASE 2: Fine-tune the selected model

Now fine-tune the best model so it directly outputs short_shelf / medium_shelf / non_perishable,
instead of relying on keyword mapping.

## Step 9: Build HuggingFace Dataset

In [10]:
from datasets import Dataset, DatasetDict

def load_image(path):
    return Image.open(path).convert("RGB")

print("Loading train images...")
train_dataset = Dataset.from_dict({
    "image": [load_image(p) for p in all_train_paths],
    "label": all_train_labels,
})

print("Loading test images...")
test_dataset = Dataset.from_dict({
    "image": [load_image(p) for p in all_test_paths],
    "label": all_test_labels,
})

split = train_dataset.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({
    "train": split["train"],
    "validation": split["test"],
    "test": test_dataset,
})

print(f"Train:      {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test:       {len(dataset['test'])}")

Loading train images...
Loading test images...
Train:      2218
Validation: 247
Test:       2112


## Step 10: Prepare data for selected model

In [11]:
from transformers import AutoImageProcessor

processor = AutoImageProcessor.from_pretrained(SELECTED_MODEL_PATH)

def preprocess(batch):
    images = [img.convert("RGB") for img in batch["image"]]
    inputs = processor(images=images, return_tensors="pt")
    inputs["label"] = batch["label"]
    return inputs

dataset["train"].set_transform(preprocess)
dataset["validation"].set_transform(preprocess)
dataset["test"].set_transform(preprocess)

print(f"Transform set for {SELECTED_MODEL_KEY}")

Transform set for Swin-tiny


## Step 11: Load model (3-class head)

In [12]:
from transformers import AutoModelForImageClassification

model = AutoModelForImageClassification.from_pretrained(
    SELECTED_MODEL_PATH,
    num_labels=len(LABEL_NAMES),
    id2label={i: l for i, l in enumerate(LABEL_NAMES)},
    label2id={l: i for i, l in enumerate(LABEL_NAMES)},
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model: {SELECTED_MODEL_KEY} | Params: {total_params:,}")

Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

SwinForImageClassification LOAD REPORT from: microsoft/swin-tiny-patch4-window7-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([3])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([3, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


Model: Swin-tiny | Params: 27,521,661


## Step 12: Train

In [13]:
from transformers import TrainingArguments, Trainer
import evaluate

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir=f"./shelf-life-{SELECTED_MODEL_KEY.lower()}",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    remove_unused_columns=False,
    fp16=True,
    dataloader_num_workers=2,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
)

train_start = time.time()
trainer.train()
train_time = time.time() - train_start
print(f"\nTraining complete in {train_time/60:.1f} minutes")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.511673,0.018701,0.991903
2,0.017642,0.003875,1.000000
3,0.002028,0.002577,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete in 1.3 minutes


## Step 13: Evaluate — Accuracy

In [14]:
ft_results = trainer.evaluate(dataset["test"])
print(f"Test Accuracy: {ft_results['eval_accuracy']:.4f}")
print(f"Test Loss:     {ft_results['eval_loss']:.4f}")

Test Accuracy: 0.9976
Test Loss:     0.0085


## Step 14: Evaluate — Precision, Recall, F1, Confusion Matrix

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

pipe_ft = hf_pipeline("image-classification", model=model, image_processor=processor, device=0)

y_true = []
y_pred = []

for i, path in enumerate(all_test_paths):
    img = Image.open(path).convert("RGB")
    true_label = LABEL_NAMES[all_test_labels[i]]
    pred = pipe_ft(img, top_k=1)
    y_true.append(true_label)
    y_pred.append(pred[0]["label"])

print("Classification Report:")
print(classification_report(y_true, y_pred, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred, labels=LABEL_NAMES))

Classification Report:
                precision    recall  f1-score   support

  medium_shelf     0.9966    0.9949    0.9957       586
non_perishable     1.0000    1.0000    1.0000        70
   short_shelf     0.9979    0.9986    0.9983      1456

      accuracy                         0.9976      2112
     macro avg     0.9982    0.9978    0.9980      2112
  weighted avg     0.9976    0.9976    0.9976      2112

Confusion Matrix:
[[1454    2    0]
 [   3  583    0]
 [   0    0   70]]


## Step 15: Inference speed

In [16]:
inf_times = []
for path in all_test_paths[:100]:
    img = Image.open(path).convert("RGB")
    t0 = time.time()
    pipe_ft(img)
    inf_times.append(time.time() - t0)

ft_avg_ms = np.mean(inf_times) * 1000
print(f"Avg inference: {ft_avg_ms:.1f}ms per image")

Avg inference: 28.5ms per image


## Step 16: Before vs After comparison

In [17]:
selected_phase1 = next(r for r in pretrained_results if r["Model"] == SELECTED_MODEL_KEY)

comparison = pd.DataFrame([
    {
        "Stage": "Pre-trained + keyword mapping",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": selected_phase1["Accuracy (pre-trained)"],
        "Avg Inference (ms)": selected_phase1["Avg Inference (ms)"],
    },
    {
        "Stage": "Fine-tuned (3-class direct)",
        "Model": SELECTED_MODEL_KEY,
        "Accuracy": round(ft_results["eval_accuracy"], 4),
        "Avg Inference (ms)": round(ft_avg_ms, 1),
    },
])

print("="*60)
print("Before vs After Fine-tuning")
print("="*60)
print(comparison.to_string(index=False))

Before vs After Fine-tuning
                        Stage     Model  Accuracy  Avg Inference (ms)
Pre-trained + keyword mapping Swin-tiny    0.6955                22.5
  Fine-tuned (3-class direct) Swin-tiny    0.9976                28.5


## Step 17: Export Excel

In [18]:
with pd.ExcelWriter("P1_Experimental_results.xlsx") as writer:
    df_selection.to_excel(writer, sheet_name="P1 Model Selection", index=False)
    comparison.to_excel(writer, sheet_name="P1 Fine-tune Result", index=False)

print("Saved!")
from google.colab import files
files.download("P1_Experimental_results.xlsx")

Saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 18: Push to HuggingFace Hub

In [19]:
HUB_MODEL_ID = "Alisa-Sun/shelf-life-classification"  # TODO: Change if needed

model.push_to_hub(HUB_MODEL_ID, commit_message=f"Fine-tuned {SELECTED_MODEL_KEY} | acc={ft_results['eval_accuracy']:.4f}")
processor.push_to_hub(HUB_MODEL_ID)

print(f"Model pushed to: https://huggingface.co/{HUB_MODEL_ID}")

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1f19v9r/model.safetensors:   0%|          |  131kB /  110MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed to: https://huggingface.co/Alisa-Sun/shelf-life-classification


## Step 19: Final inference test

In [20]:
pipe_final = hf_pipeline("image-classification", model=HUB_MODEL_ID)

print("Testing 5 images:")
for i in range(min(5, len(all_test_paths))):
    img = Image.open(all_test_paths[i]).convert("RGB")
    true_label = LABEL_NAMES[all_test_labels[i]]
    pred = pipe_final(img)
    status = "✅" if pred[0]["label"] == true_label else "❌"
    print(f"  {status} True: {true_label} | Pred: {pred[0]['label']} ({pred[0]['score']:.3f})")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

Testing 5 images:
  ✅ True: short_shelf | Pred: short_shelf (1.000)
  ✅ True: short_shelf | Pred: short_shelf (1.000)
  ✅ True: short_shelf | Pred: short_shelf (0.999)
  ✅ True: short_shelf | Pred: short_shelf (1.000)
  ✅ True: short_shelf | Pred: short_shelf (1.000)


---
## Summary

| Step | Content |
|------|---------|
| 1-5 | Setup + download Grocery Store + Household Products → build 3-class dataset |
| 6-8 | **Phase 1:** Pre-trained + keyword mapping comparison (no training) → select best |
| 9-12 | **Phase 2:** Fine-tune selected model (3 epochs, full data) |
| 13-15 | Evaluate: accuracy, precision, recall, confusion matrix, speed |
| 16 | Before (keyword mapping) vs After (fine-tuned) comparison |
| 17 | Export Excel |
| 18-19 | Push to Hub + final test |

**Key difference from Pipeline 2:** Phase 1 here uses keyword mapping to compare models (no training needed), while Pipeline 2 used quick 1-epoch fine-tuning. Both approaches are valid for model selection.